In [4]:
import numpy as np
from PIL import Image
from scipy.io import wavfile

## Задание 1. Фурье-преобразование картинки.

Сделайте преобразование Фурье от картинки в файле `im1.jpg`, используя файл `rian.py`. Сохраните двумерный фурье-образ в файле `out.jpg` (параметры цвета r, g, b округлять до целых чисел). Удалите из фурье-образа верхнюю половину пространственных частот по каждой координате. Сделайте обратное преобразование и сохраните в файле `out2.jpg`

**Указание.**
Методом двумерного преобразования Фурье (two-dimensional Fourier transform - 2-DFT) является преобразование Фурье, произведенное над двумерным массивом данных. Рассмотрим двумерный массив данных. Эти данные имеют два измерения: x и y. Преобразование Фурье над данными производится сначала в одном, а затем в другом направлениях.

Первая часть преобразования Фурье проводится в x измерении. Вторая часть преобразования Фурье производится в y измерении.

In [3]:
img = Image.open('im1.jpg').convert('RGB')
data = np.array(img).astype(float)
h, w, c = data.shape

f_shifted_full = np.zeros((h, w, c), dtype=complex)
data_filtered = np.zeros((h, w, c))
spectrum_vis = np.zeros((h, w, c))

for i in range(c):
    f = np.fft.fft2(data[:, :, i])
    f_shifted = np.fft.fftshift(f)
    f_shifted_full[:, :, i] = f_shifted

    mag = np.abs(f_shifted)
    spec = 255 * (np.log(1 + mag) / np.log(1 + np.max(mag)))
    spectrum_vis[:, :, i] = spec

    f_filtered = f_shifted.copy()
    cy, cx = h // 2, w // 2
    rh, rw = h // 4, w // 4

    mask = np.zeros((h, w))
    mask[cy-rh:cy+rh, cx-rw:cx+rw] = 1
    f_filtered *= mask

    f_ishifted = np.fft.ifftshift(f_filtered)
    img_back = np.fft.ifft2(f_ishifted)
    data_filtered[:, :, i] = np.clip(np.real(img_back), 0, 255)

out_img = Image.fromarray(np.uint8(np.round(spectrum_vis)))
out_img.save('out.jpg')

out2_img = Image.fromarray(np.uint8(np.round(data_filtered)))
out2_img.save('out2.jpg')

## Задание 2. Фурье-преобразование звука.

Возьмите файл `in10.wav` и, используя `sound.py`, добавьте в его фурье-образ низкочастотный шум. Сохраните результат в `out1.wav`. После этого обрежьте нижние частоты в фурье-образе и сохраните в `out2.wav`.

In [5]:
sample_rate, data = wavfile.read('in10.wav')
if data.dtype == np.int16:
    data = data.astype(np.float32) / 32768.0

channels = data.shape[1] if len(data.shape) > 1 else 1
spectrum = np.fft.rfft(data, axis=0)
N = spectrum.shape[0]

low_freq_limit = int(N * 0.02)
noise_magnitude = np.max(np.abs(spectrum)) * 0.5
noise = (np.random.normal(0, noise_magnitude, (low_freq_limit, channels)) +
         1j * np.random.normal(0, noise_magnitude, (low_freq_limit, channels)))

spectrum_noisy = spectrum.copy()
spectrum_noisy[:low_freq_limit] += noise

out1_data = np.fft.irfft(spectrum_noisy, axis=0)
out1_data = np.clip(out1_data, -1, 1) # ограничение амплитуды
wavfile.write('out1.wav', sample_rate, (out1_data * 32767).astype(np.int16))

spectrum_filtered = spectrum_noisy.copy()
spectrum_filtered[:low_freq_limit] = 0

out2_data = np.fft.irfft(spectrum_filtered, axis=0)
out2_data = np.clip(out2_data, -1, 1)
wavfile.write('out2.wav', sample_rate, (out2_data * 32767).astype(np.int16))

## Задание 3*. Быстрое умножение многочленов.

Напишите алгоритм быстрого умножения многочленов с помощью БПФ.

Более подробное описание тут умножение многочленов или тут алгоритм

Проверьте, использует ли его Sympy, сравнив скорость работы.